# Exploring Mini Data Split

In [1]:
import xarray as xr

area_path = "data/northamerica-west-mini/"
data_in_area = xr.open_zarr(area_path + "train_data_in.zarr")
data_out_area = xr.open_zarr(area_path + "train_data_out.zarr")

In [2]:
data_in_area

<xarray.Dataset> Size: 46MB
Dimensions:    (time: 100, latitude: 80, longitude: 80)
Coordinates:
  * time       (time) datetime64[ns] 800B 2001-01-01 ... 2001-01-05T03:00:00
  * latitude   (latitude) float32 320B 60.0 59.75 59.5 ... 40.75 40.5 40.25
  * longitude  (longitude) float32 320B 230.0 230.2 230.5 ... 249.2 249.5 249.8
Data variables:
    cape       (time, latitude, longitude) float64 5MB ...
    cp         (time, latitude, longitude) float64 5MB ...
    sp         (time, latitude, longitude) float64 5MB ...
    tclw       (time, latitude, longitude) float64 5MB ...
    tcw        (time, latitude, longitude) float64 5MB ...
    tisr       (time, latitude, longitude) float64 5MB ...
    tp         (time, latitude, longitude) float64 5MB ...
    u          (time, latitude, longitude) float64 5MB ...
    v          (time, latitude, longitude) float64 5MB ...
Attributes:
    Conventions:  CF-1.6
    history:      2024-06-05 08:11:26 GMT by grib_to_netcdf-2.28.1: /opt/ecmw...

In [ ]:
# Get the start and end of your time grid
start_date = data_in_area.coords['time'].min().values
end_date = data_in_area.coords['time'].max().values
print("start", start_date)
print("end", end_date)

# Get the resolution
lat_step = data_in_area.coords['latitude'].diff('latitude').mean().values
lat_min = data_in_area.coords['latitude'].min().values
lat_max = data_in_area.coords['latitude'].max().values

lon_min = data_in_area.coords['longitude'].min().values
lon_max = data_in_area.coords['longitude'].max().values

print("lat_step", lat_step)

print("lat range", lat_min, "to", lat_max)
print("lon range", lon_min, "to", lon_max)

start 2001-01-01T00:00:00.000000000
end 2001-01-05T03:00:00.000000000
lat_step -0.25
lat range 40.25 to 60.0
lon range 230.0 to 249.75


In [8]:
data_out_area

<xarray.Dataset> Size: 16MB
Dimensions:        (time: 100, latitude: 200, longitude: 200)
Coordinates:
  * latitude       (latitude) float32 800B 59.95 59.85 59.75 ... 40.15 40.05
  * longitude      (longitude) float32 800B 230.1 230.2 230.2 ... 249.9 250.0
Dimensions without coordinates: time
Data variables:
    precipitation  (time, latitude, longitude) float32 16MB ...

In [9]:
# Get the start and end of your time grid
start_date = data_out_area.coords['time'].min().values
end_date = data_out_area.coords['time'].max().values
print("start", start_date)
print("end", end_date)

# Get the resolution (e.g., is it every 1 degree or 5 degrees?)
lat_step = data_out_area.coords['latitude'].diff('latitude').mean().values
lat_min = data_out_area.coords['latitude'].min().values
lat_max = data_out_area.coords['latitude'].max().values

lon_min = data_out_area.coords['longitude'].min().values
lon_max = data_out_area.coords['longitude'].max().values

print("lat_step", lat_step)

print("lat range", lat_min, "to", lat_max)
print("lon range", lon_min, "to", lon_max)

start 0
end 99
lat_step -0.09999999
lat range 40.05 to 59.949997
lon range 230.05 to 249.95001


# RainShift Dataset Inspection
Get coordinate bounds for each RainShift region and the sampling resolution and save it to a csv.

In [ ]:
import pandas as pd
import numpy as np
import xarray as xr

def get_coord_bounds(data_array):
    # adjust the longitude range to be -180 to 180 instead of 0 to 360
    data_array.coords['longitude'] = (data_array.coords['longitude'] + 180) % 360 - 180

    lat_min = data_array.coords['latitude'].min().values
    lat_max = data_array.coords['latitude'].max().values
    lon_min = data_array.coords['longitude'].min().values
    lon_max = data_array.coords['longitude'].max().values
        
    return lat_min, lat_max, lon_min, lon_max

def get_time_bounds(data_array):
    start_date = data_array.coords['time'].min().values
    end_date = data_array.coords['time'].max().values
    return start_date, end_date

def get_resolution(data_array):
    lat_step = data_array.coords['latitude'].diff('latitude').mean().values
    lon_step = data_array.coords['longitude'].diff('longitude').mean().values
    return lat_step, lon_step


In [ ]:
# Dataset configuration
REPO_ID = "RainShift/rainshift"
REGIONS = [
    "africa-south",
    "amazon-basin",
    "arabian-peninsula",
    "australasia-east",
    "blacksea",
    "cape-horn",
    "caribbean",
    "east-asia-north-east",
    "east-asia-south",
    "europe_west",
    "horn-of-africa",
    "melanesia",
    "northamerica-east",
    "northamerica-west",
    "southamerica-east",
    "southeastasia-west",
    "tibetan-plateau",
    "west-africa"
]

SPLITS = ["train_data_in", "train_data_out", "test_data_in", "test_data_out"]
SPLIT = "test_data_out" # smallest split for now

regions_info = {}

for region in REGIONS:
    ds = xr.open_zarr(f"data/rainshift_dataset/{region}/{SPLIT}.zarr")
    lat_min, lat_max, lon_min, lon_max = get_coord_bounds(ds)
    start_date, end_date = get_time_bounds(ds)
    lat_step, lon_step = get_resolution(ds)

    # round 
    lat_min_rounded = np.round(lat_min, 0)
    lat_max_rounded = np.round(lat_max, 0)
    lon_min_rounded = np.round(lon_min, 0)
    lon_max_rounded = np.round(lon_max, 0)
    lat_step_rounded = np.round(lat_step, 1)
    lon_step_rounded = np.round(lon_step, 1)

    assert np.abs(lat_max_rounded - lat_min_rounded) == 20, f"Expected latitude range of 20 degrees, but got {lat_max, lat_min} for region {region}"
    assert np.abs(lon_max_rounded - lon_min_rounded) == 20, f"Expected longitude range of 20 degrees, but got {lon_max, lon_min} for region {region}"

    regions_info[region] = {
        "lat_min": lat_min_rounded,
        "lat_max": lat_max_rounded,
        "lon_min": lon_min_rounded,
        "lon_max": lon_max_rounded,
        "start_date": start_date,
        "end_date": end_date,
        "lat_step": lat_step_rounded,
        "lon_step": lon_step_rounded
    }
    # print(f"Region: {region}")
    # print(f"  Time range: {start_date} to {end_date}")
    # print(f"  Latitude range: {lat_min_rounded} to {lat_max_rounded}")
    # print(f"  Longitude range: {lon_min_rounded} to {lon_max_rounded}")

regions_df = pd.DataFrame.from_dict(regions_info, orient='index').reset_index(names='region')
print(regions_df)
regions_df.to_csv("outputs/rainshift_regions_info.csv", index=False)


                  region  lat_min  lat_max  lon_min  lon_max start_date  \
0           africa-south    -40.0    -20.0     18.0     38.0          0   
1           amazon-basin    -20.0     -0.0    -60.0    -40.0          0   
2      arabian-peninsula     15.0     35.0     40.0     60.0          0   
3       australasia-east    -40.0    -20.0    137.0    157.0          0   
4               blacksea     35.0     55.0     22.0     42.0          0   
5              cape-horn    -56.0    -36.0    -80.0    -60.0          0   
6              caribbean      5.0     25.0    -80.0    -60.0          0   
7   east-asia-north-east     30.0     50.0    130.0    150.0          0   
8        east-asia-south     20.0     40.0    100.0    120.0          0   
9            europe_west     40.0     60.0     -5.0     15.0          0   
10        horn-of-africa     -5.0     15.0     32.0     52.0          0   
11             melanesia    -15.0      5.0    130.0    150.0          0   
12     northamerica-east 